# LLM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split


In [ ]:
# Load the saved model and tokenizer
model = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/Ddrive")
tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/Ddrive")

# Now you can use the loaded model and tokenizer for inference or fine-tuning

In [ ]:


ml_df1 = pd.read_csv('/content/drive/MyDrive/Ddrive/ml_data_beta.csv')


def scale_dataset(data, oversample=False):
    # Convert datetime column to numerical representation
    #data['Date'] = pd.to_numeric(data['Date'])

    # Separate features and target variable
    X = data.drop(columns=['Injured'])
    y = data['Injured']

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=104)

    # Oversample the training set if specified
    # Oversample the training set using SMOTE
    smote = SMOTE(random_state=104)
    X_train, y_train = smote.fit_resample(X_train, y_train)

    # Scale the features
    #scaler = StandardScaler()
    #X_train_scaled = scaler.fit_transform(X_train)
    #X_test_scaled = scaler.transform(X_test)

    return (X_train, y_train), (X_test, y_test)



# because data is much and would take longer to train
sample_size = 5000
ml_df1 = ml_df1.sample(n=sample_size, random_state=104)


(X_train, y_train), (X_test, y_test) = scale_dataset(ml_df1, oversample=True)


print("Shape of x_train :", X_train.shape)
print("Shape of y_train :", y_train.shape)
print("Shape of x_test :", X_test.shape)
print("Shape of y_test :", y_test.shape)

# Check the distribution of the target class in each dataset
train_class_distribution = y_train.value_counts()
test_class_distribution = y_test.value_counts()

print("Distribution of the target class in training set:")
print(train_class_distribution)

print("\nDistribution of the target class in testing set:")
print(test_class_distribution)


Shape of x_train : (7920, 13)
Shape of y_train : (7920,)
Shape of x_test : (1000, 13)
Shape of y_test : (1000,)
Distribution of the target class in training set:
Injured
0    3960
1    3960
Name: count, dtype: int64

Distribution of the target class in testing set:
Injured
0    990
1     10
Name: count, dtype: int64


In [ ]:
X_test['Injured'] = y_test
X_test.to_excel('/content/drive/MyDrive/Ddrive/x_test_data.xlsx')

In [ ]:
# Create LLM prompt

def concatenate_text(x):
    full_text = (
        f"This player performance during training in position {x['Position Name']} shows that, ",
        f"he ran a distance of {x['Distance/minute']}, ",
        f"with HSR speed of {x['HSR/minute']} metres/min, ",
        f"and  sprinting speed {x['Sprinting/minute']}. ",
        f"The player completed {x['Explosive Efforts']} efforts, ",
        f"covering {x['Acceleration B3 Efforts (Gen 2)']} metres accelerating, ",
        f"and {x['Deceleration B3 Efforts (Gen 2)']} metres decelerating. " ,
        f"He has a Player load of {x['Total Player Load']}. "
        f"The energy usage by the player was {x['EDI']}, "
        f"with the profile max velocity being {x['Profile Max Velocity']}, "
        f"and completing {x['ACC + DEC/minute']} number of accelerations per minute of the training session"


    )
    return ''.join(full_text)




In [ ]:
X_train['label'] = y_train
X_test['label'] = y_test

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['text'].iloc[0]

'This player performance during training in position 4.0 shows that, he ran a distance of 88.10211946050097, with HSR speed of 3.82466281310212 metres/min, and  sprinting speed 0.9826589595375724. The player completed 18.0 efforts, covering 1.0 metres accelerating, and 5.0 metres decelerating. He has a Player load of 844.0. The energy usage by the player was 1.13256, with the profile max velocity being 32.76, and completing 0.3853564547206166 number of accelerations per minute of the training session'

In [ ]:
import torch
# Take a sample from the test data
sample_test_data = X_test.sample(n=1)  # Assuming X_test is your test data

# Prepare the text input
text_input = sample_test_data['text'].iloc[0]  # Extract the text from the sample

# Tokenize the text input
inputs = tokenizer(text_input, return_tensors='pt', padding=True, truncation=True)

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted label
predicted_label = torch.argmax(outputs.logits).item()

# Define label mappings
id2label = {0: "NOT-INJURED", 1: "INJURED"}
label2id = {"NOT-INJURED": 0, "INJURED": 1}

# Map the predicted label to the corresponding class
predicted_class = id2label[predicted_label]

print("Predicted Class:", predicted_class)

Predicted Class: NOT-INJURED


In [ ]:

# Now you can deploy your loaded model and tokenizer for inference or fine-tuning
sample_test_data = X_test.sample(n=3)  # Assuming X_test is your test data

# Prepare the text input
text_input = sample_test_data['text'].iloc[1]
# Tokenize the text input
inputs = tokenizer(text_input, return_tensors='pt', padding=True, truncation=True)

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted label
predicted_label = torch.argmax(outputs.logits).item()


# Define label mappings
id2label = {0: "NOT-INJURED", 1: "INJURED"}
label2id = {"NOT-INJURED": 0, "INJURED": 1}

# Map the predicted label to the corresponding class
predicted_class = id2label[predicted_label]

print("Predicted Class:", predicted_class)

Predicted Class: NOT-INJURED


In [ ]:
X_test

,PlayerID,RecurrenceOfPreviousInjury,Position Name,Distance/minute,HSR/minute,Sprinting/minute,Total Player Load,ACC + DEC/minute,Acceleration B3 Efforts (Gen 2),Deceleration B3 Efforts (Gen 2),Profile Max Velocity,Explosive Efforts,EDI,Injured,label,text
7784,6,0,11.0,47.076326,1.578266,0.129366,364,0.439845,5.0,2.0,34.56000,24,1.19746,0,0,This player performance during training in pos...
12311,144,0,3.0,75.551664,2.661996,0.963222,451,0.385289,1.0,4.0,32.04000,18,1.14000,0,0,This player performance during training in pos...
7961,63,0,15.0,100.029557,8.433498,2.630542,957,0.876847,11.0,12.0,38.00016,47,1.17403,0,0,This player performance during training in pos...
764,52,0,6.0,72.256944,1.302083,0.121528,416,0.381944,0.0,1.0,30.96000,21,1.14100,0,0,This player performance during training in pos...
5548,65,0,4.0,57.267206,0.728745,0.000000,287,0.263158,0.0,2.0,33.58800,6,1.13616,0,0,This player performance during training in pos...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11950,50,0,14.0,74.518167,4.470774,1.216430,554,0.663507,0.0,6.0,32.76000,29,1.19400,0,0,This player performance during training in pos...
5296,77,0,1.0,52.771855,11.599147,0.639659,209,0.063966,0.0,1.0,33.48000,6,1.08705,0,0,This player performance during training in pos...
14139,4,0,3.0,114.709351,5.787700,0.935131,1454,0.547599,2.0,5.0,32.40000,49,1.14000,0,0,This player performance during training in pos...
2814,21,0,4.0,94.004796,4.572342,0.247802,1091,0.751399,8.0,8.0,34.92000,58,1.17000,0,0,This player performance during training in pos...


In [ ]:
sample_df = X_test
# Load test data (X_test, y_test)
# Assuming X_test is a list of text inputs and y_test is a list of corresponding labels
sample_x_df = sample_df['text'].tolist()
# Prepare test inputs
test_inputs = tokenizer(sample_x_df, return_tensors='pt', padding=True, truncation=True)

# Perform inference
with torch.no_grad():
    outputs = model(**test_inputs)


# Get predicted labels
predicted_labels = torch.argmax(outputs.logits, dim=1).tolist()



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_tests = sample_df['label']
# Calculate evaluation metrics
accuracy = accuracy_score(y_tests, predicted_labels)
precision = precision_score(y_tests, predicted_labels, average='weighted')
recall = recall_score(y_tests, predicted_labels, average='weighted')
f1 = f1_score(y_tests, predicted_labels, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.988
Precision: 0.9800801603206413
Recall: 0.988
F1-score: 0.9840241448692152


In [ ]:
class InjuryPredictor:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def load_data(self, file_path):
        # Load GPS data from Excel file
        gps = pd.read_excel(file_path)

        # Set the first row as column names
        #new_header = gps.iloc[0]
        #gps = gps[1:]
        #gps.columns = new_header

        return gps

    def preprocess_data(self, gps_data, player_id, period_number=0):
        # Filter data for the specified player ID and period number

        player_data = gps_data[(gps_data['PlayerID'] == player_id) & (gps_data['Period Number'] == period_number)]

        # encode label
        le = LabelEncoder()
        player_data["Position Name"] = le.fit_transform(player_data["Position Name"])

        # Calculate additional features
        player_data['Distance/minute'] = player_data['Total Distance'] / player_data['Duration (minutes)']
        player_data['HSR/minute'] = player_data['HSR & Sprinting'] / player_data['Duration (minutes)']
        player_data['Sprinting/minute'] = player_data['Sprinting'] / player_data['Duration (minutes)']
        player_data['Total Player Load/minute'] = player_data['Total Player Load'] / player_data['Duration (minutes)']
        player_data['ACC + DEC/minute'] = player_data['ACC + DEC'] / player_data['Duration (minutes)']

        # Select useful columns
        useful_columns = ['Position Name', 'Distance/minute', 'HSR/minute', 'Sprinting/minute',
                           'Total Player Load', 'ACC + DEC/minute',
                           'Acceleration B3 Efforts (Gen 2)', 'Deceleration B3 Efforts (Gen 2)',
                           'Profile Max Velocity', 'Explosive Efforts', 'EDI']

        player_data = player_data[useful_columns]

        # Calculate the mean of each column
        column_means = player_data.mean()

        # Add a new column containing the concatenated text for average performance
        player_data['Performance Summary'] = self.concatenate_text(column_means)

        return player_data

    def predict_injury(self, player_data):
        # Prepare the text input
        text_input = player_data['Performance Summary'].iloc[0]

        # Tokenize the text input
        inputs = self.tokenizer(text_input, return_tensors='pt', padding=True, truncation=True)

        # Perform inference
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Get the predicted label
        predicted_label = torch.argmax(outputs.logits).item()

        # Define label mappings
        id2label = {0: "NOT-INJURED", 1: "INJURED"}
        label2id = {"NOT-INJURED": 0, "INJURED": 1}

        # Map the predicted label to the corresponding class
        predicted_class = id2label[predicted_label]

        return predicted_class

    def concatenate_text(self, x):
        full_text = (
            f"This player performance during training in position {x['Position Name']} shows that, ",
            f"he ran a distance of {x['Distance/minute']}, ",
            f"with HSR speed of {x['HSR/minute']} metres/min, ",
            f"and  sprinting speed {x['Sprinting/minute']}. ",
            f"The player completed {x['Explosive Efforts']} efforts, ",
            f"covering {x['Acceleration B3 Efforts (Gen 2)']} metres accelerating, ",
            f"and {x['Deceleration B3 Efforts (Gen 2)']} metres decelerating. " ,
            f"He has a Player load of {x['Total Player Load']}. "
            f"The energy usage by the player was {x['EDI']}, "
            f"with the profile max velocity being {x['Profile Max Velocity']}, "
            f"and completing {x['ACC + DEC/minute']} number of accelerations per minute of the training session"


        )
        return ''.join(full_text)


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder

# Instantiate the InjuryPredictor class with your model and tokenizer instances
injury_predictor = InjuryPredictor(model, tokenizer)

# Load GPS data from Excel file
file_path = '/content/drive/MyDrive/Ddrive/validation_data_beta.xlsx'
gps_data = injury_predictor.load_data(file_path)

# Preprocess data for PlayerID 3
player_id = 5
player_data = injury_predictor.preprocess_data(gps_data, player_id)

# Predict whether the player will be injured or not
predicted_class = injury_predictor.predict_injury(player_data)

print("Predicted Class:", predicted_class)

Predicted Class: INJURED


In [ ]:
ml_df1.columns

Index(['PlayerID', 'RecurrenceOfPreviousInjury', 'Position Name',
       'Distance/minute', 'HSR/minute', 'Sprinting/minute',
       'Total Player Load', 'ACC + DEC/minute',
       'Acceleration B3 Efforts (Gen 2)', 'Deceleration B3 Efforts (Gen 2)',
       'Profile Max Velocity', 'Explosive Efforts', 'EDI', 'Injured'],
      dtype='object')

# XGB

In [ ]:
# Import the required functions
from xgboost import XGBClassifier
import xgboost as xgb
import joblib
# Load the saved model (PICKLE)
#loaded_model = joblib.load('xgb_model.pkl')

# Load the saved model
loaded_model = xgb.XGBClassifier()
loaded_model.load_model('/content/drive/MyDrive/Ddrive/xgb_model.json')

In [ ]:
class XGBPredictor:
    def __init__(self, loaded_model):
        self.model = loaded_model
        self.label_map = {0: "NOT INJURED", 1: "INJURED"}

    def load_data(self, file_path):
        # Load GPS data from Excel file
        gps = pd.read_excel(file_path)
        # Set the first row as column names
        #new_header = gps.iloc[0]
        #gps = gps[1:]
        #gps.columns = new_header
        return gps

    def preprocess_data(self, gps_data, player_id, period_number=0):
        # Filter data for the specified player ID and period number
        player_data = gps_data[(gps_data['PlayerID'] == player_id) & (gps_data['Period Number'] == period_number)]
        # encode label
        le = LabelEncoder()
        player_data["Position Name"] = le.fit_transform(player_data["Position Name"])
        # Calculate additional features
        player_data['Distance/minute'] = player_data['Total Distance'] / player_data['Duration (minutes)']
        player_data['HSR/minute'] = player_data['HSR & Sprinting'] / player_data['Duration (minutes)']
        player_data['Sprinting/minute'] = player_data['Sprinting'] / player_data['Duration (minutes)']
        player_data['Total Player Load/minute'] = player_data['Total Player Load'] / player_data['Duration (minutes)']
        player_data['ACC + DEC/minute'] = player_data['ACC + DEC'] / player_data['Duration (minutes)']
        # Select useful columns
        useful_columns = ['PlayerID', 'RecurrenceOfPreviousInjury', 'Position Name', 'Distance/minute', 'HSR/minute', 'Sprinting/minute',
                          'Total Player Load', 'ACC + DEC/minute', 'Acceleration B3 Efforts (Gen 2)',
                          'Deceleration B3 Efforts (Gen 2)', 'Profile Max Velocity', 'Explosive Efforts', 'EDI']
        player_data = player_data[useful_columns]
        # Add a new column 'RecurrenceOfPreviousInjury' with all values set to 0
        #player_data = player_data[useful_columns].assign(RecurrenceOfPreviousInjury=0)
        # Calculate the mean of each column
        player_data = player_data.mean()
        return player_data

    def predict_injury(self, player_data):
        # Convert player data to a format suitable for prediction
        player_data = player_data.values.reshape(1, -1)

        # Make the prediction
        prediction = self.model.predict(player_data)

        # Map the predicted value to the corresponding label
        predicted_label = self.label_map.get(prediction[0], "Unknown")

        return predicted_label

In [ ]:
# Create an instance of the XGBPredictor class
predictor = XGBPredictor(loaded_model)

# Load and preprocess the data
file_path = '/content/drive/MyDrive/Ddrive/validation_data_beta.xlsx'
gps_data = predictor.load_data(file_path)
player_id = 5
period_number = 0
player_data = predictor.preprocess_data(gps_data, player_id, period_number)

# Make the prediction
predicted_injury = predictor.predict_injury(player_data)
print(f"Predicted injury status: {predicted_injury}")

Predicted injury status: INJURED
